In [1]:
from utils.util import *
from shapely.geometry import Point
fileType = 'band'
bandNames = {'SR_B4', 'SR_B5', 'ST_B10'}
includeMetadata = True
point = Point(-149.8555, 61.2433)

Directory 'Data' already exists.
Directory 'utils' already exists.
Logging in...


Login Successful, API Key Received!


In [2]:
features = []
features = [Feature(geometry=point, properties={"city": "Anchorage", "state": "Alaska"})]

feature_collection = FeatureCollection(features)

with open('./utils/Anchorage_Alaska_aoi.geojson', 'w') as f:
    dump(feature_collection, f)
    
aoi_geodf =  gpd.read_file('./utils/Anchorage_Alaska_aoi.geojson') #aoi geopandas dataframe
aoi_geodf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [3]:
import folium
m = folium.Map(location=[aoi_geodf.geometry.y[0], aoi_geodf.geometry.x[0]], zoom_start=12, tiles="openstreetmap", 
              width="90%", height="90%", attributionControl=0)

In [4]:
folium.Marker([aoi_geodf.geometry.y[0], aoi_geodf.geometry.x[0]], popup="Anchorage").add_to(m)
# m

In [5]:
datasetName = 'landsat_ot_c2_l2'

In [6]:
# Corrected spatial filter using a small circular area
spatialFilter = {
    'filterType': 'circle',
    'centerPoint': {
        'latitude': aoi_geodf.geometry.y[0],
        'longitude': aoi_geodf.geometry.x[0]
    },
    'radius': 0.01  # Small radius in kilometers
}

temporalFilter = {'start' : '2020-03-01', 'end' : '2020-03-15'}
cloudCoverFilter = {'min' : 0, 'max' : 20}
search_payload = {
    'datasetName' : datasetName,
    'sceneFilter' : {
        'spatialFilter' : spatialFilter,
        'acquisitionFilter' : temporalFilter,
        'cloudCoverFilter' : cloudCoverFilter
    }
}
search_payload

{'datasetName': 'landsat_ot_c2_l2',
 'sceneFilter': {'spatialFilter': {'filterType': 'circle',
   'centerPoint': {'latitude': 61.2433, 'longitude': -149.8555},
   'radius': 0.01},
  'acquisitionFilter': {'start': '2020-03-01', 'end': '2020-03-15'},
  'cloudCoverFilter': {'min': 0, 'max': 20}}}

In [7]:
scenes = sendRequest(serviceUrl + "scene-search", search_payload, apiKey)
pd.json_normalize(scenes['results'])

,browse,cloudCover,entityId,displayId,orderingId,metadata,hasCustomizedMetadata,publishDate,options.bulk,options.download,...,options.secondary,selected.bulk,selected.compare,selected.order,spatialBounds.type,spatialBounds.coordinates,spatialCoverage.type,spatialCoverage.coordinates,temporalCoverage.endDate,temporalCoverage.startDate
0,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",9,LC80680172020070LGN00,LC08_L2SP_068017_20200310_20200822_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2022-06-22 18:27:47-05,True,True,...,False,False,False,False,Polygon,"[[[-150.79515, 60.35665], [-150.79515, 62.5793...",Polygon,"[[[-150.79515, 60.91032], [-147.47326, 60.3566...",2020-03-10 00:00:00,2020-03-10 00:00:00


In [8]:
idField = 'entityId'
entityIds = []
for result in scenes['results']:
    if result['options']['bulk'] == True:
        entityIds.append(result[idField])
entityIds

['LC80680172020070LGN00']

In [9]:
listId = f"temp_{datasetName}_list" # customized list id
scn_list_add_payload = {
    "listId": listId,
    'idField' : idField,
    "entityIds": entityIds,
    "datasetName": datasetName
}
scn_list_add_payload

{'listId': 'temp_landsat_ot_c2_l2_list',
 'idField': 'entityId',
 'entityIds': ['LC80680172020070LGN00'],
 'datasetName': 'landsat_ot_c2_l2'}

In [10]:
count = sendRequest(serviceUrl + "scene-list-add", scn_list_add_payload, apiKey) 
count

1

In [11]:
sendRequest(serviceUrl + "scene-list-get", {'listId' : scn_list_add_payload['listId']}, apiKey) 

[{'entityId': 'LC80680172020070LGN00', 'datasetName': 'landsat_ot_c2_l2'}]

In [12]:
download_opt_payload = {
    "listId": listId,
    "datasetName": datasetName
}

if fileType == 'band_group':
    download_opt_payload['includeSecondaryFileGroups'] = True

download_opt_payload

{'listId': 'temp_landsat_ot_c2_l2_list', 'datasetName': 'landsat_ot_c2_l2'}

In [13]:
products = sendRequest(serviceUrl + "download-options", download_opt_payload, apiKey)
pd.json_normalize(products)

,id,downloadName,displayId,entityId,datasetId,available,filesize,productName,productCode,bulkAvailable,downloadSystem,secondaryDownloads,fileGroups
0,5e83d14fec7cae84,None,LC08_L2SP_068017_20200310_20200822_02_T1,LC80680172020070LGN00,5e83d14f2fc39685,True,977600762,Landsat Collection 2 Level-2 Product Bundle,D694,True,ls_zip,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
1,6448198cc7b442a4,C2L2 Tile Product Files,LC08_L2SP_068017_20200310_20200822_02_T1,LC80680172020070LGN00,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D693,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
2,6448198c62023764,C2L2 Tile Product Files,LC08_L2SP_068017_20200310_20200822_02_T1,LC80680172020070LGN00,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D691,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
3,632210d4770592cf,None,LC08_L2SP_068017_20200310_20200822_02_T1,LC80680172020070LGN00,5e83d14f2fc39685,False,977600762,Landsat Collection 2 Level-2 Product Bundle,D806,False,dds_ms,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None


In [14]:
filegroups = sendRequest(serviceUrl + "dataset-file-groups", {'datasetName' : datasetName}, apiKey)  
pd.json_normalize(filegroups['secondary'])

,5f85f041c828327a.ls_c2l2_sr_band.id,5f85f041c828327a.ls_c2l2_sr_band.color,5f85f041c828327a.ls_c2l2_sr_band.description,5f85f041c828327a.ls_c2l2_sr_band.displayOrder,5f85f041c828327a.ls_c2l2_sr_band.icon,5f85f041c828327a.ls_c2l2_sr_band.label,5f85f041c828327a.ls_c2l2_sr_band.files,5f85f041c828327a.ls_c2l2_st_band.id,5f85f041c828327a.ls_c2l2_st_band.color,5f85f041c828327a.ls_c2l2_st_band.description,5f85f041c828327a.ls_c2l2_st_band.displayOrder,5f85f041c828327a.ls_c2l2_st_band.icon,5f85f041c828327a.ls_c2l2_st_band.label,5f85f041c828327a.ls_c2l2_st_band.files
0,ls_c2l2_sr_band,None,Landsat Collection-2 Level-2 Surface Reflectan...,1,None,Level-2 Surface Reflectance Bands,"[{'name': 'ANG.txt', 'productIds': {'6453e8a2f...",ls_c2l2_st_band,None,Landsat Collection-2 Level-2 Surface Temperatu...,1,None,Level-2 Surface Temperature Bands,"[{'name': 'ANG.txt', 'productIds': {'6453e8a2f..."


In [15]:
fileGroupIds = {"ls_c2l2_sr_band"}

In [16]:
downloads = []
print("    Selecting band files...")
for product in products:
    if product["secondaryDownloads"] is not None and len(product["secondaryDownloads"]) > 0:
        for secondaryDownload in product["secondaryDownloads"]:
            for bandName in bandNames:
                if secondaryDownload["bulkAvailable"] and bandName in secondaryDownload['displayId']:
                    downloads.append({"entityId": secondaryDownload["entityId"], "productId": secondaryDownload["id"]})
            # Check for the metadata file (_MTL.txt)
            if includeMetadata and secondaryDownload["displayId"].endswith('_MTL.txt'):
                downloads.append({"entityId": secondaryDownload["entityId"], "productId": secondaryDownload["id"]})

    Selecting band files...


In [17]:
download_req2_payload = {
    "downloads": downloads,
    "label": label
}

print(f"Sending download request ...")
download_request_results = sendRequest(serviceUrl + "download-request", download_req2_payload, apiKey)
print(f"Done sending download request") 

if len(download_request_results['newRecords']) == 0 and len(download_request_results['duplicateProducts']) == 0:
    print('No records returned, please update your scenes or scene-search filter')
    sys.exit()

Sending download request ...
Done sending download request


In [18]:
# Attempt the download URLs
for result in download_request_results['availableDownloads']:
    print(f"Get download url: {result['url']}\n" )
    runDownload(threads, result['url'])
    
preparingDownloadCount = len(download_request_results['preparingDownloads'])
preparingDownloadIds = []
if preparingDownloadCount > 0:
    for result in download_request_results['preparingDownloads']:  
        preparingDownloadIds.append(result['downloadId'])

    download_ret_payload = {"label" : label}                
    # Retrieve download URLs
    print("Retrieving download urls...\n")
    download_retrieve_results = sendRequest(serviceUrl + "download-retrieve", download_ret_payload, apiKey, False)
    if download_retrieve_results != False:
        print(f"    Retrieved: \n" )
        for result in download_retrieve_results['available']:
            if result['downloadId'] in preparingDownloadIds:
                preparingDownloadIds.remove(result['downloadId'])
                runDownload(threads, result['url'])
                print(f"       {result['url']}\n" )
            
        for result in download_retrieve_results['requested']:   
            if result['downloadId'] in preparingDownloadIds:
                preparingDownloadIds.remove(result['downloadId'])
                runDownload(threads, result['url'])
                print(f"       {result['url']}\n" )
    
    # Didn't get all download URLs, retrieve again after 30 seconds
    while len(preparingDownloadIds) > 0: 
        print(f"{len(preparingDownloadIds)} downloads are not available yet. Waiting for 30s to retrieve again\n")
        time.sleep(30)
        download_retrieve_results = sendRequest(serviceUrl + "download-retrieve", download_ret_payload, apiKey, False)
        if download_retrieve_results != False:
            for result in download_retrieve_results['available']:                            
                if result['downloadId'] in preparingDownloadIds:
                    preparingDownloadIds.remove(result['downloadId'])
                    print(f"    Get download url: {result['url']}\n" )
                    runDownload(threads, result['url'])
                    
print("\nDownloading files... Please do not close the program\n")
for thread in threads:
    thread.join()        

Get download url: https://landsatlook.usgs.gov/data/collection02/level-2/standard/oli-tirs/2020/068/017/LC08_L2SP_068017_20200310_20200822_02_T1/LC08_L2SP_068017_20200310_20200822_02_T1_MTL.txt?requestSignature=eyJkb3dubG9hZEFwcCI6Ik0yTSIsImNvbnRhY3RJZCI6MjczNjI1NzcsImRvd25sb2FkSWQiOjcwODc5MTA0MywiZGF0ZUdlbmVyYXRlZCI6IjIwMjQtMTItMDdUMTU6MjM6MTItMDY6MDAiLCJpZCI6IkxDMDhfTDJTUF8wNjgwMTdfMjAyMDAzMTBfMjAyMDA4MjJfMDJfVDFfTVRMLnR4dCIsInNpZ25hdHVyZSI6IiQ1JCRjUXg5ZXlVc2x1TXpTRlwvYUZcL2gzR2FDdFVpOXMwLlJRTTBsUFJQdGI0YTkifQ==

Get download url: https://landsatlook.usgs.gov/data/collection02/level-2/standard/oli-tirs/2020/068/017/LC08_L2SP_068017_20200310_20200822_02_T1/LC08_L2SP_068017_20200310_20200822_02_T1_SR_B4.TIF?requestSignature=eyJkb3dubG9hZEFwcCI6Ik0yTSIsImNvbnRhY3RJZCI6MjczNjI1NzcsImRvd25sb2FkSWQiOjcwODc5MTA0NiwiZGF0ZUdlbmVyYXRlZCI6IjIwMjQtMTItMDdUMTU6MjM6MTItMDY6MDAiLCJpZCI6IkxDMDhfTDJTUF8wNjgwMTdfMjAyMDAzMTBfMjAyMDA4MjJfMDJfVDFfU1JfQjQuVElGIiwic2lnbmF0dXJlIjoiJDUkJHNcL1dXMXg0d0hxMWhmTV

In [19]:
remove_scnlst_payload = {
    "listId": listId
}
sendRequest(serviceUrl + "scene-list-remove", remove_scnlst_payload, apiKey)

In [20]:
endpoint = "logout"  
if sendRequest(serviceUrl + endpoint, None, apiKey) == None:        
    print("\nLogged Out\n")
else:
    print("\nLogout Failed\n")


Logged Out



In [21]:
os.listdir(data_dir)

['LC08_L2SP_068017_20200310_20200822_02_T1_MTL.txt',
 'LC08_L2SP_068017_20200310_20200822_02_T1_SR_B4.TIF',
 'LC08_L2SP_068017_20200310_20200822_02_T1_SR_B5.TIF',
 'LC08_L2SP_068017_20200310_20200822_02_T1_ST_B10.TIF']